# EdAi Project — Compression study: **resnet18** (trained from scratch, 100 epochs)

Which lightweight method keeps a malaria screener **clinically safe** on a Raspberry Pi?

| Variant | What it does | Expected effect on a Pi CPU |
|---|---|---|
| FP32 | baseline | reference |
| INT8 (minmax / percentile) | 8-bit weights + activations (post-training) | ~4x smaller, often faster on ARM |
| Unstructured pruning 50% | zero the smallest 50% of weights + fine-tune | smaller after gzip, **not faster** |
| Structured pruning 30% / 50% | remove whole channels + fine-tune | fewer FLOPs → **faster** |
| Structured pruning + INT8 | both combined | smallest + fastest (if accuracy holds) |

Every variant is calibrated on validation (own temperature + LED threshold) and tested on 30 unseen patients.
Model metrics **and** system metrics are recorded; all ONNX files are saved for the Pi benchmark.

**Run with:** Add Input → *Malaria Cell Images Dataset* · GPU · Internet **On** · then **Save Version → Save & Run All**.

## 0. Get the code from GitHub
Code lives in **[https://github.com/Vemuri-BK/edge-malaria-screening](https://github.com/Vemuri-BK/edge-malaria-screening)**.
Change code on your laptop → `git push` → re-run this notebook. Each run records the commit it used.

*Private repo?* Add a Kaggle secret named `GITHUB_TOKEN` (Add-ons → Secrets) with a GitHub personal access token.

In [ ]:
import os, sys, glob, json, shutil, subprocess

REPO_URL = "https://github.com/Vemuri-BK/edge-malaria-screening.git"
BRANCH = "main"

ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working/edge-malaria-screening" if ON_KAGGLE else os.path.abspath("edge-malaria-screening")

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(TOKEN, "***") if TOKEN else r.stderr
        raise RuntimeError("git " + args[0] + " failed: " + msg)
    return r.stdout.strip()

# Optional token for a private repo (Kaggle secret GITHUB_TOKEN)
TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = REPO_URL.replace("https://", "https://" + TOKEN + "@") if TOKEN else REPO_URL

if os.path.isdir(os.path.join(WORK, ".git")):          # re-run: update to latest
    git("fetch", "-q", "origin", BRANCH, cwd=WORK)
    git("reset", "-q", "--hard", "origin/" + BRANCH, cwd=WORK)
else:
    git("clone", "-q", "--depth", "1", "-b", BRANCH, url, WORK)

os.chdir(WORK)
os.makedirs("outputs", exist_ok=True)
GIT_COMMIT = git("log", "-1", "--format=%h %s (%cd)", "--date=short")
print("Code version:", GIT_COMMIT)
print(git("ls-files", "src"))

# onnx / onnxruntime are needed for export + quantization
try:
    import onnx, onnxruntime
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)

def find_data_root(base="/kaggle/input"):
    # folder that contains Parasitized/ and Uninfected/ (shallowest copy)
    hits = sorted(glob.glob(os.path.join(base, "**", "Parasitized"), recursive=True),
                  key=lambda p: p.count(os.sep))
    if not hits:
        raise FileNotFoundError("Malaria dataset not found - add it with 'Add Input'.")
    return os.path.dirname(hits[0])

DATA_ROOT = find_data_root()
print("Working dir:", WORK)
print("Data root  :", DATA_ROOT)

## 1. Imports, extra packages, system information

In [ ]:
try:
    import torch_pruning
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-pruning"], check=True)

sys.path.insert(0, os.path.join(WORK, "src"))
import time
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

import config as C
from dataset import build_loaders, load_split, build_image_cache, cache_to_input
from models import EdgeClassifier, count_parameters
from sysmon import system_info
from train import set_seed, fit
from export_onnx import export_onnx, check_parity
from quantize import quantize_int8
from compress import (unstructured_prune, make_pruning_permanent, structured_prune,
                      count_flops_params, sparsity)
from study import evaluate_variant, recommend

SYS_TRAIN = system_info()
for k, v in SYS_TRAIN.items():
    print(f"{k:20s}: {v}")

## 2. Study settings

In [ ]:
EXP = "study_resnet18_scratch"
C.ARCH = "resnet18"
C.PRETRAINED = False   # False = trained from scratch (random initial weights)
C.EPOCHS = 100           # baseline training
C.LR = 0.003
FT_EPOCHS = 5                 # fine-tuning after pruning
UNSTRUCTURED_AMOUNT = 0.5     # fraction of weights set to zero
STRUCTURED_RATIOS = [0.3, 0.5]  # fraction of channels removed

OUT = os.path.join(WORK, "outputs", EXP)
DEPLOY = os.path.join(WORK, "outputs", "deploy_" + EXP)
os.makedirs(OUT, exist_ok=True); os.makedirs(DEPLOY, exist_ok=True)
print(f"{EXP}: baseline {C.EPOCHS} epochs, fine-tune {FT_EPOCHS} epochs after pruning")

## 3. Data: patient-level split + images cached in RAM
Decoding PNGs was the training bottleneck (GPU ~15% busy). Each image is decoded and resized once; pixels are identical to the Pi's preprocessing.

In [ ]:
!python src/data_prep.py --data_root "{DATA_ROOT}"
set_seed()
SPLIT_CSV = os.path.join(WORK, "splits", "patient_split.csv")
parts = load_split(SPLIT_CSV)
t0 = time.time()
caches = {s: build_image_cache(df, DATA_ROOT) for s, df in parts.items()}
print(f"Cached {sum(len(c) for c in caches.values())} images in {time.time() - t0:.0f}s "
      f"({sum(c.nbytes for c in caches.values()) / 1e9:.2f} GB)")
loaders, _ = build_loaders(SPLIT_CSV, DATA_ROOT, caches=caches)
Xv, yv = cache_to_input(caches["val"]), parts["val"].label.values
Xt, yt = cache_to_input(caches["test"]), parts["test"].label.values
Xcal = cache_to_input(caches["train"][np.random.default_rng(C.SEED).choice(len(caches["train"]), C.CALIB_IMAGES, replace=False)])

## 4. Baseline FP32 model
Re-uses the baseline only if a previous training run **finished** (e.g. after a kernel restart); an interrupted run is trained again.

In [ ]:
base_dir = os.path.join(OUT, "baseline")
model = EdgeClassifier(C.ARCH, pretrained=C.PRETRAINED)
ckpt = os.path.join(base_dir, "best_model.pth")
if os.path.exists(os.path.join(base_dir, "COMPLETE")):
    model.load_state_dict(torch.load(ckpt, map_location="cpu"))
    hist = pd.read_csv(os.path.join(base_dir, "history.csv"))
    print("Loaded existing baseline")
else:
    model, hist = fit(model, loaders, base_dir, epochs=C.EPOCHS, lr=C.LR)
BASE_TRAIN_MIN = hist.epoch_time_s.sum() / 60
print(f"Baseline training time: {BASE_TRAIN_MIN:.1f} min, best val AUROC {hist.val_auroc.max():.4f}")
hist[["epoch", "train_loss", "val_loss", "val_auroc", "epoch_time_s", "train_throughput_img_s", "gpu_util_mean"]].round(4)

## 5. Build and evaluate every variant

In [ ]:
rows, models_cfg, train_minutes = [], {}, {"FP32": BASE_TRAIN_MIN}
led_ref = None

def add(name, onnx_path, torch_model=None, method=""):
    global led_ref
    extra = {"method": method}
    if torch_model is not None:
        mflops, params = count_flops_params(torch_model)
        extra.update(params_m=params / 1e6, mflops=mflops, sparsity=sparsity(torch_model))
    extra["extra_train_min"] = train_minutes.get(name, 0.0)
    row, led = evaluate_variant(name, onnx_path, Xv, yv, Xt, yt, led_reference=led_ref, extra=extra)
    if name == "FP32":
        led_ref = led
    rows.append(row)
    models_cfg[name] = {"file": os.path.basename(onnx_path), "temperature": row["temperature"],
                        "led_threshold": row["led_threshold"]}
    print(f"{name:32s} AUROC {row['auroc']:.4f} sens {row['sensitivity']:.4f} "
          f"FN {row['fn']:4d} FP {row['fp']:4d} | {row['onnx_mb']:.2f} MB | {row['cpu_latency_ms_1thread']:.2f} ms")

def export(m, name):
    path = os.path.join(DEPLOY, name + ".onnx")
    export_onnx(m, path)
    assert check_parity(m, path) < 1e-3
    return path

def int8(src_path, name, method="percentile"):
    path = os.path.join(DEPLOY, name + ".onnx")
    quantize_int8(src_path, path, Xcal, method=method)
    return path

# --- FP32 baseline + post-training quantization
fp32_path = export(model, "fp32")
add("FP32", fp32_path, model, "baseline")
add("INT8 minmax", int8(fp32_path, "int8_minmax", "minmax"), method="quantization")
add("INT8 percentile", int8(fp32_path, "int8_percentile"), method="quantization")

# --- Unstructured pruning (+ fine-tune), then + INT8
t0 = time.time()
m_u = unstructured_prune(model, UNSTRUCTURED_AMOUNT)
m_u, _ = fit(m_u, loaders, os.path.join(OUT, "unstructured"), epochs=FT_EPOCHS, lr=C.LR / 10)
make_pruning_permanent(m_u)
name = f"Unstructured {int(UNSTRUCTURED_AMOUNT * 100)}%"
train_minutes[name] = (time.time() - t0) / 60
p_u = export(m_u, "unstructured")
add(name, p_u, m_u, "pruning")
add(name + " + INT8", int8(p_u, "unstructured_int8"), method="pruning + quantization")

# --- Structured channel pruning (+ fine-tune), then + INT8
for r in STRUCTURED_RATIOS:
    t0 = time.time()
    m_s = structured_prune(model, r)
    m_s, _ = fit(m_s, loaders, os.path.join(OUT, f"structured_{int(r * 100)}"), epochs=FT_EPOCHS, lr=C.LR / 2)
    name = f"Structured {int(r * 100)}%"
    train_minutes[name] = (time.time() - t0) / 60
    p_s = export(m_s, f"structured_{int(r * 100)}")
    add(name, p_s, m_s, "pruning")
    add(name + " + INT8", int8(p_s, f"structured_{int(r * 100)}_int8"), method="pruning + quantization")

results = pd.DataFrame(rows).set_index("variant")
results.to_csv(os.path.join(OUT, "study_results.csv"))
cols = ["method", "params_m", "mflops", "sparsity", "onnx_mb", "gzip_mb", "auroc", "accuracy",
        "sensitivity", "specificity", "ece", "fn", "fp", "led_flips_vs_fp32", "cpu_latency_ms_1thread", "extra_train_min"]
results[cols].round(4)

## 6. Trade-offs and deployment recommendation
Rule: among variants as safe as FP32 (sensitivity within 1 point, AUROC within 0.005), choose the fastest. **Latency here is Kaggle's x86 CPU — confirm on the Pi with `pi/benchmark.py`.**

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for v, r in results.iterrows():
    mk = "s" if "INT8" in v else "o"
    ax[0].scatter(r.cpu_latency_ms_1thread, r.sensitivity, marker=mk, s=60); ax[0].annotate(v, (r.cpu_latency_ms_1thread, r.sensitivity), fontsize=8)
    ax[1].scatter(r.onnx_mb, r.auroc, marker=mk, s=60); ax[1].annotate(v, (r.onnx_mb, r.auroc), fontsize=8)
ax[0].axhline(results.loc["FP32", "sensitivity"] - 0.01, ls="--", c="gray")
ax[0].set_xlabel("CPU latency (ms, 1 thread)"); ax[0].set_ylabel("test sensitivity"); ax[0].set_title("Safety vs speed")
ax[1].set_xlabel("ONNX size (MB)"); ax[1].set_ylabel("test AUROC"); ax[1].set_title("Quality vs size")
results["fn"].plot.barh(ax=ax[2], color="tab:red"); ax[2].set_title("Missed infections (FN) on test"); ax[2].invert_yaxis()
plt.tight_layout(); plt.savefig(os.path.join(OUT, "study_tradeoffs.png"), dpi=120); plt.show()

safe, best = recommend(results)
print("Clinically safe variants (fastest first):")
print(safe[["onnx_mb", "auroc", "sensitivity", "fn", "cpu_latency_ms_1thread"]].round(4))
print(f"\nRecommended for the Pi (provisional): {best}")

## 7. Save everything for the laptop / Pi benchmark

In [ ]:
deploy_config = {
    "experiment": EXP,
    "arch": C.ARCH,
    "classes": C.CLASSES,
    "positive_class_index": 1,
    "img_size": C.IMG_SIZE,
    "input": "RGB, resized to img_size x img_size (bilinear), float32 in [0,1], NCHW; normalization is inside the model",
    "led_rule": "LED ON if softmax(logits / temperature)[1] >= led_threshold (per model)",
    "target_sensitivity": C.TARGET_SENSITIVITY,
    "recommended": best,
    "models": models_cfg,
    # backwards-compatible keys (used by older scripts)
    "model_fp32": models_cfg["FP32"]["file"], "temperature_fp32": models_cfg["FP32"]["temperature"],
    "model_int8": models_cfg["INT8 percentile"]["file"], "led_threshold": models_cfg["FP32"]["led_threshold"],
}
with open(os.path.join(DEPLOY, "deploy_config.json"), "w") as f:
    json.dump(deploy_config, f, indent=2)

demo_dir = os.path.join(DEPLOY, "demo_images"); os.makedirs(demo_dir, exist_ok=True)
demo = pd.concat([parts["test"][parts["test"].label == l].sample(10, random_state=1) for l in (0, 1)])
for p in demo.path:
    shutil.copy(os.path.join(DATA_ROOT, p), os.path.join(demo_dir, os.path.basename(p)))
demo.assign(file=demo.path.map(os.path.basename))[["file", "class", "label", "patient_id"]].to_csv(
    os.path.join(demo_dir, "labels.csv"), index=False)

with open(os.path.join(OUT, "study_summary.json"), "w") as f:
    json.dump({"experiment": EXP, "code_version": GIT_COMMIT, "system_training": SYS_TRAIN,
               "recommended": best, "results": results.reset_index().to_dict("records")}, f, indent=2, default=float)

zip_base = os.path.join("/kaggle/working" if ON_KAGGLE else WORK, "edai_" + EXP)
shutil.make_archive(zip_base, "zip", os.path.join(WORK, "outputs"))
print(f"Saved. Download {os.path.basename(zip_base)}.zip from the Output panel.")